In [1]:
# Mengimpor library untuk manipulasi data, regex, ekstraksi fitur TF-IDF (Scikit-Learn), dan stopword (Sastrawi).
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

In [2]:
# Mendefinisikan korpus mentah yang berisi kumpulan dokumen teks untuk diolah.
korpus_mentah = [
    "Jaringan komputer lokal berfungsi menghubungkan berbagai perangkat keras.",
    "Kecerdasan buatan dan machine learning saat ini berkembang sangat pesat.",
    "Perangkat keras dan perangkat lunak membentuk fondasi sistem komputer.",
    "Sistem temu kembali informasi memudahkan proses pencarian data teks."
]

In [3]:
# Fungsi preprocessing untuk membersihkan teks (case folding, regex) dan menghapus stopword, lalu menerapkannya pada korpus.
daftar_stopword = StopWordRemoverFactory().get_stop_words()

def normalisasi_teks(teks):
    teks = teks.lower()
    teks_bersih = re.sub(r'[^a-z\s]', ' ', teks)
    kumpulan_kata = [kata for kata in teks_bersih.split() if kata not in daftar_stopword]
    return ' '.join(kumpulan_kata)

dokumen_siap_olah = [normalisasi_teks(kalimat) for kalimat in korpus_mentah]
print("Teks Setelah Preprocessing:\n", dokumen_siap_olah, "\n")

Teks Setelah Preprocessing:
 ['jaringan komputer lokal berfungsi menghubungkan berbagai perangkat keras', 'kecerdasan buatan machine learning berkembang sangat pesat', 'perangkat keras perangkat lunak membentuk fondasi sistem komputer', 'sistem temu informasi memudahkan proses pencarian data teks'] 



In [4]:
# Melakukan ekstraksi kosa kata unik dan menghitung bobot TF-IDF secara manual tahap demi tahap hingga membentuk matriks.
kosa_kata_unik = sorted(list(set(' '.join(dokumen_siap_olah).split())))
jumlah_dok = len(dokumen_siap_olah)
label_baris = [f"Dokumen-{i+1}" for i in range(jumlah_dok)]

data_frekuensi = []
for dok in dokumen_siap_olah:
    token = dok.split()
    data_frekuensi.append({kata: token.count(kata) for kata in kosa_kata_unik})

matriks_tf = pd.DataFrame(data_frekuensi, index=label_baris)

distribusi_df = (matriks_tf > 0).sum(axis=0)
idf_penyesuaian = np.log((1 + jumlah_dok) / (1 + distribusi_df)) + 1

tf_idf_kalkulasi = matriks_tf * idf_penyesuaian
tf_idf_manual = tf_idf_kalkulasi.div(np.linalg.norm(tf_idf_kalkulasi, axis=1), axis=0)

print("=== MATRIKS TF-IDF (PERHITUNGAN MANUAL) ===")
display(tf_idf_manual)

=== MATRIKS TF-IDF (PERHITUNGAN MANUAL) ===


,berbagai,berfungsi,berkembang,buatan,data,fondasi,informasi,jaringan,kecerdasan,keras,...,memudahkan,menghubungkan,pencarian,perangkat,pesat,proses,sangat,sistem,teks,temu
Dokumen-1,0.381669,0.381669,0.000000,0.000000,0.000000,0.000000,0.000000,0.381669,0.000000,0.300912,...,0.000000,0.381669,0.000000,0.300912,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Dokumen-2,0.000000,0.000000,0.377964,0.377964,0.000000,0.000000,0.000000,0.000000,0.377964,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.377964,0.000000,0.377964,0.000000,0.000000,0.000000
Dokumen-3,0.000000,0.000000,0.000000,0.000000,0.000000,0.368827,0.000000,0.000000,0.000000,0.290787,...,0.000000,0.000000,0.000000,0.581574,0.000000,0.000000,0.000000,0.290787,0.000000,0.000000
Dokumen-4,0.000000,0.000000,0.000000,0.000000,0.362224,0.000000,0.362224,0.000000,0.000000,0.000000,...,0.362224,0.000000,0.362224,0.000000,0.000000,0.362224,0.000000,0.285582,0.362224,0.362224


In [5]:
# Menghitung bobot TF-IDF secara otomatis dengan library Scikit-Learn dan memvalidasi keakuratan hitungan manual.
mesin_vektor = TfidfVectorizer(vocabulary=kosa_kata_unik)
ekstraksi_sklearn = mesin_vektor.fit_transform(dokumen_siap_olah)
tf_idf_sklearn = pd.DataFrame(ekstraksi_sklearn.toarray(), columns=kosa_kata_unik, index=label_baris)

print("\n=== MATRIKS TF-IDF (LIBRARY SCIKIT-LEARN) ===")
display(tf_idf_sklearn)

margin_error = (tf_idf_manual - tf_idf_sklearn).abs().max().max()
print(f"\nMargin Error Maksimal Manual vs Sklearn: {margin_error}")


=== MATRIKS TF-IDF (LIBRARY SCIKIT-LEARN) ===


,berbagai,berfungsi,berkembang,buatan,data,fondasi,informasi,jaringan,kecerdasan,keras,...,memudahkan,menghubungkan,pencarian,perangkat,pesat,proses,sangat,sistem,teks,temu
Dokumen-1,0.381669,0.381669,0.000000,0.000000,0.000000,0.000000,0.000000,0.381669,0.000000,0.300912,...,0.000000,0.381669,0.000000,0.300912,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Dokumen-2,0.000000,0.000000,0.377964,0.377964,0.000000,0.000000,0.000000,0.000000,0.377964,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.377964,0.000000,0.377964,0.000000,0.000000,0.000000
Dokumen-3,0.000000,0.000000,0.000000,0.000000,0.000000,0.368827,0.000000,0.000000,0.000000,0.290787,...,0.000000,0.000000,0.000000,0.581574,0.000000,0.000000,0.000000,0.290787,0.000000,0.000000
Dokumen-4,0.000000,0.000000,0.000000,0.000000,0.362224,0.000000,0.362224,0.000000,0.000000,0.000000,...,0.362224,0.000000,0.362224,0.000000,0.000000,0.362224,0.000000,0.285582,0.362224,0.362224



Margin Error Maksimal Manual vs Sklearn: 0.0


**Analisis TF-IDF:** Metode pembobotan TF-IDF (Term Frequency-Inverse Document Frequency) mengevaluasi seberapa penting sebuah kata dalam suatu dokumen relatif terhadap kumpulan dokumen (korpus). Kata yang sering muncul di satu dokumen (TF tinggi) tetapi jarang di dokumen lain (IDF tinggi) akan mendapatkan bobot tinggi, menjadikannya kata kunci penentu (diskriminan). Dalam konteks sistem temu kembali informasi (Information Retrieval), perhitungan TF-IDF secara signifikan meningkatkan akurasi pencarian. Sistem tidak sekadar mengandalkan jumlah kemunculan kata yang bisa saja didominasi oleh kata umum, tetapi juga melihat keunikannya, sehingga dokumen relevan dapat diurutkan dengan tingkat *precision* yang jauh lebih baik.